![NWM](../img/NWM.png)

# Use HydroData to Retrieve Modeled and Observed Snow Data for a Watershed of Interest 
## ParFlow-CONUS Outputs vs Observed Snow Water Equivalent (SWE) 

Authors: Irene Garousi-Nejad (igarousi@cuahsi.org), Danielle Tijerina-Kreuzer (dtijerina@cuahsi.org)  
Last updated: April 2026

#### Introduction:  
This notebook demonstrates how to perform a point-scale analysis comparing modeled and observed SWE at selected SNOTEL sites, implementing a full Model Evaluation Workflow. 
We focus on analyzing model performance both for **a single SNOTEL site** and **watershed-scale behavior for multiple stations**, with particular attention to the **magnitude and timing of peak SWE**.   

## 1. Setup

### 1a. Python Environment  

Ensure that the `cssi_evaluation` conda environment is selected as your Jupyter kernel. This environment should already be created if you followed the instructions under section "Creating your HydroLearnEnv Virtual Environment" in the `GettingStarted.md` file.

Import the libraries needed to run this notebook:

In [ ]:
import os
import sys
from pathlib import Path
import holoviews as hv
import hvplot.pandas
import hvplot.xarray
import plotly.io as pio
import plotly.express as px
import pyproj
import pandas as pd
import numpy as np
import xarray as xr
import geopandas as gpd
from dask.distributed import Client
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import hf_hydrodata as hf
import subsettools


# Import the Evaluation library from the project root.
sys.path.append(str((Path.cwd().absolute() / "../../src").resolve()))

from cssi_evaluation.variables import snow_utils
from cssi_evaluation.utils import metric_utils, evaluation_utils, plot_utils

hv.extension('bokeh')


%load_ext autoreload
%autoreload 2

### 1b. Register Pin and Access HydroData

To access the HydroData catalog you will need to sign up for a [HydroFrame account](https://hydrogen.princeton.edu/signup) (do this only once), [create a 4-digit PIN](https://hydrogen.princeton.edu/pin), and register your pin in order to have access to the HydroData datasets (you will do this in the next code cell below). To note, you PIN will expire after 7 days and will need to recreate it after that time. 

In [ ]:
# You need to register on https://hydrogen.princeton.edu/pin 
# and run the following with your registered information
# before you can use the hydrodata utilities
#hf.register_api_pin("your_email", "your_api_pin")
hf.register_api_pin("dtt2@princeton.edu", "7837")

### 1c. Dask  

We'll use dask to parallelize our code. To manage parallel computation and visualize progress of long-running tasks, we initialize a Dask “cluster,” which defines how many workers are used and how much computing power each worker has. 

In this setup, we create a Dask client with `Client(n_workers=6, threads_per_worker=1, memory_limit='2GB')`, which launches a cluster with 6 workers. Each worker uses a single thread, typically mapped to one CPU core, allowing for efficient parallel processing across 6 cores. Each worker also has a memory limit of 2 GB, for a total of up to 12 GB across the cluster.


In [ ]:
# use a try accept loop so we only instantiate the client
# if it doesn't already exist.
try:
    print('Dashboard link:', client.dashboard_link)
except:    
    # The client should be customized to your workstation resources.
    client = Client(n_workers=6, threads_per_worker=1, memory_limit='2GB') 
    print('Dashboard link:', client.dashboard_link)
print(client)

### 1d. Set Paths

In [ ]:
# Start and end times of a water year (to note, these dates were chosen to align with the PFCONUS1 early 2000s runs)
StartDate = '2003-10-01'
EndDate = '2005-09-30'

domain_data_path = './domain_data/' # path to the model domain data

# Path to save results (obs and mod stands for observation and modeled, respectively)
OBS_OutputFolder = './obs_outputs' 
MOD_OutputFolder = './mod_outputs'

## 2. Retrieve SNOTEL point observations and metadata from HydroData  

### 2a. Define the watershed of interest

One of the simplest ways to gather data and model output from HydroData is by specifying a [Hydrologic Unit Code](https://www.usgs.gov/national-hydrography/watershed-boundary-dataset). Before we retrieve any hydrologic information, we need to indicate a HUC8 code and use it to gather snow water equivalent (SWE) observations from SNOTEL sites  

✏️ If you have a specific HUC8 in mind, you can change the variable `huc_8_code` below.

In [ ]:
# ✏️ Specify HUC8 ID and Name for watershed of interest
huc_8_code = '14010001'  # East-Taylor HUC-8
print(f'HUC-8 ID: {huc_8_code}')

huc_8_name = 'Colorado-Headwaters'
print(f'HUC-8 name: {huc_8_name}')

### 2b. Explore the available SWE data in a watershed  

Explore what SWE data is available at SNOTEL sites within the HUC ID you specified that operated during WY2004 and WY2005.

<div style="color:black;background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
<h4>📖 Did you know?</h4>
<p>The Snow Telemetry (SNOTEL) network, managed by the USDA Natural Resources Conservation Service (NRCS), monitors snowpack conditions across key watersheds in the western United States to support water supply forecasting and climate monitoring. SNOTEL sites are fully automated stations that continuously measure snow water equivalent (SWE), snow depth, precipitation, temperature, and other meteorological variables throughout the year. Unlike manual snow survey programs, SNOTEL provides high-temporal-resolution observations that enable near–real-time assessment of snowpack evolution and interannual variability. These data are widely used for operational forecasting, drought assessment, and long-term climate analysis. </p>
</div>

In [ ]:
# Create a folder to save observations
Path(OBS_OutputFolder).mkdir(exist_ok=True)

#### Retrieve the metadata for the SNOTEL stations in the watershed:

In [ ]:
# Request site-level attributes for these sites
metadata_df = hf.get_point_metadata(dataset="snotel", variable="swe", temporal_resolution="daily", aggregation="sod",
                                 date_start=StartDate, date_end=EndDate,
                                 huc_id=[huc_8_code], grid='conus1')

# save
metadata_df.to_csv(f'./{OBS_OutputFolder}/df_{huc_8_name}_{huc_8_code}_SNOTEL_metadata.csv', index=False)

# View first five records
metadata_df.head(5)

The **metadata file** in HydroData is an important addition to the observations and it is recommended to always gather and save this for the observations you are using (particularly to support reproducibility within an open-science workflow). The saved file has useful attributes like site names, first and last date of available data, lat/lon, and the query URL.  

>Additionally, the metadata contains **ParFlow-CONUS1 and ParFlow-CONUS2 `i,j` indices, which indicate the exact model domain grid cell the observation aligns with**. This is a useful HydroData feature that removes the need for users to manually match station latitude/longitude coordinates to the appropriate model grid cell, as this spatial mapping is handled directly within HydroData. We will use these indices below to extract PF-CONUS1 modeled SWE for each SNOTEL station in the section below. 

#### Map the SNOTEL stations inside the HUC-08 watershed that have available data within the selected time range   

In [ ]:
# Let's define a helper plotting function to use throughout the notebook
def plot_sites_on_map(metadata_df, color_by_site_type=False):
    """
    Plots site locations on an interactive map using Plotly.
    
    Parameters:
    - metadata_df: DataFrame containing site metadata with 'latitude', 'longitude', 'site_name', and 'site_id' columns.
    - color_by_site_type: Boolean indicating whether to color points by site type.
    """
    pio.renderers.default = "notebook"

    center = {
        "lat": metadata_df["latitude"].mean(),
        "lon": metadata_df["longitude"].mean()
    }

    fig = px.scatter_map(
        metadata_df,
        lat="latitude",
        lon="longitude",
        color="site_type" if color_by_site_type else None,
        hover_name="site_name",      # appears on hover
        hover_data=["site_id"],      # additional info
        center=center,
        zoom=8,
        height=600
    )

    fig.update_layout(mapbox_style="carto-voyager")
    fig.update_traces(marker=dict(size=12))
    return fig

Plot these sites on a map. Then, hover over the pins to see the site names.

In [ ]:
plot_sites_on_map(metadata_df, color_by_site_type=False)


### 2c. Gather the SNOTEL data for all stations within the watershed  
Here we use the HydroData `hf.get_point_data()` function to retrieve daily, start-of-day SWE from SNOTEL sites:

In [ ]:
# Request point observations data
obs_df = hf.get_point_data(dataset="snotel", variable="swe", temporal_resolution="daily", aggregation="sod",
                         date_start=StartDate, date_end=EndDate,
                         huc_id=[huc_8_code], grid='conus1')
                         #polygon=watershed_bbox, polygon_crs=watershed_crs)

# save
obs_df.to_csv(f'./{OBS_OutputFolder}/df_{huc_8_name}_{huc_8_code}_SNOTEL.csv', index=False)

# Ensure date column is datetime
obs_df["date"] = pd.to_datetime(obs_df["date"])

# View first five records
obs_df.head(5)

## 3. Retrieve ParFlow-CONUS1 Modeled Snow Data

In [ ]:
# Create a folder to save results
Path(MOD_OutputFolder).mkdir(exist_ok=True)

The following section retrieves ParFlow-CONUS1 data for each SNOTEL site within our HUC-08 watershed. The code identifies the CONUS1 `i,j` indices associated with each SNOTEL site, indicated in the `metadata_df`. It then extracts the CONUS1 modeled SWE output for the site and the period of interest, returning the result as a DataFrame. To fairly compare with SNOTEL, which reports SWE once daily at the start of the local day, model output is aggregated by day, using the argment `"temporal_resolution": "daily"`. Finally, the processed data is saved as a CSV file for each site.  

### 3a. ParFlow CONUS1 Model Dataset Information
We can print some information about the model output dataset by using the `hf.get_catalog_entry()` to get the CONUS1 model dataset metadata. 

In [ ]:
conus1_options = {
    "dataset": "conus1_baseline_mod",
    "variable": "swe"
}
hf.get_catalog_entry(conus1_options)

Before we gather model outputs at the specific SNOTEL sites, we can visualize SWE across our HUC-08. This is plotted for one day at 1km lateral resolution.

In [ ]:
# retrieve gridded PF-CONUS1 SWE for the entire HUC8 watershed
grid_swe_options = {
        "dataset": "conus1_baseline_mod",
        "variable": "swe",
        "temporal_resolution": "daily",
        "start_time": '2004-04-01', ### TO NOTE: the gridded function has exclusive end date, so this is hardcoded for now 
        "end_time": '2004-04-02',
        "huc_id": huc_8_code
    }
    
    # Get gridded data
grid_swe = hf.get_gridded_data(grid_swe_options)

In [ ]:
grid_swe_map = xr.DataArray(grid_swe[0], dims=("y", "x"), name="SWE")
grid_swe_map.hvplot.image(cmap="YlGnBu", colorbar=True, aspect="equal", title=f"{huc_8_name} Gridded SWE on 2004-04-01")


Create a copy of the model dataframe (`model_df`) so we have the same data structure:

In [ ]:
# Copy obs_df to model_df so we have the same timestamps and site_id structure
model_df = obs_df.copy()

# Set all non-date columns to NaN to prepare for filling in model data
non_date_cols = model_df.columns.difference(["date"])
model_df[non_date_cols] = np.nan

# Keep official SNOTEL site names for obs columns and use same for model
model_df["date"] = pd.to_datetime(model_df["date"])

Now, retrieve the PF-CONUS1 modeled SWE from the the corresponding SNOTEL locations. Here we use the CONUS1 `i,j` indices from the `metadata_df` and grab the SWE from those grid cells using the function `hf.get_gridded_data()`:

In [ ]:
# Loop over each station in metadata_df
for idx, row in metadata_df.iterrows():
    site_id = row["site_id"]  
    col_name = site_id        

    conus_i = int(row["conus1_i"])
    conus_j = int(row["conus1_j"]) 
   
    
    # Build options dict for this station
    options = {
        "dataset": "conus1_baseline_mod",
        "variable": "swe",
        "temporal_resolution": "daily",
        "start_time": '2003-10-01', ### TO NOTE: the gridded function has exclusive end date, so this is hardcoded for now 
        "end_time": '2005-10-01',
        "grid_point": [conus_i, conus_j]
    }
    
    # Get gridded data
    data = hf.get_gridded_data(options)
    
    # Fill column in model_df
    # Convert to numeric in case hf returns lists or other types
    model_df[col_name] = np.squeeze(np.array(data))

# Ensure date column is datetime
model_df["date"] = pd.to_datetime(model_df["date"])

# Save
model_df.to_csv(f'./{MOD_OutputFolder}/df_{huc_8_name}_{huc_8_code}_PFCONUS1.csv', index=False)
    
model_df.head(5)

#### Quick sanity check plot  
Plot a simple timeseries of modeled and observed SWE to make sure our data retrieval was successful. 

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

for site_id in obs_df.columns:
    if site_id == "date":
        continue

    ax.plot(obs_df["date"], obs_df[site_id], label=f"{site_id} Obs", linewidth=2)
    ax.plot(obs_df["date"], model_df[site_id], linestyle="--", label=f"{site_id} Mod")

ax.set_xlabel("Date")
ax.set_ylabel("SWE (mm)")

# Date formatting
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%Y'))

ax.legend(loc='upper left', fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()

## 4. Compare Modeled and Observed SWE Timeseries at a Single Site

Once we have both observation data and modeling outpus, it's important to evaluate how well the model reproduces observed data. The following plots are simple timeseries comparisons of **modeled vs. observed** SWE. These types of plots provide a straight-forward visual of how well the observations and simulations agree and are a great start for assessing general model performance.  

📊 We include two figures:

1. **Time Series Overlay:** Plots the observed and modeled values together over time. This helps identify:
   - Periods of systematic bias
   - Timing differences in peaks and lows
   - General agreement in trends

2. **Scatter Plot with 1:1 Line:** Plots each modeled value against its corresponding observed value. This highlights:
   - Accuracy across the full range of values
   - Over- or under-prediction patterns
   - Outliers or extreme events

Review the sites within the watershed from the interactive map above and click on the markers to view the site name and code. Recall, we also printed out the site metadata for all sites within the watershed, which contains the SNOTEL site codes.

✏️ Once you’ve identified the site of interest, **enter its site code in the next code cell for `my_site_code`**:   

In [ ]:
# choose a site of interest within the watershed
my_site_code = '335:CO:SNTL'

# make sure date columns are datetime and set as index for easier plotting and metric calculations
obs_df["date"] = pd.to_datetime(obs_df["date"])
model_df["date"] = pd.to_datetime(model_df["date"])

obs_df = obs_df.set_index("date")
model_df = model_df.set_index("date")

Use the `comparison_plots` utility function to provide a quicky view of the data.

In [ ]:
plot_utils.comparison_plots(obs_df,  model_df, f'{my_site_code}', f'{my_site_code}', site_label=None)

To move beyond an overall summary of daily performance, we replot the modeled vs. observed SWE scatter while highlighting specific months with a distinct color. This gives us more information about the **seasonal model performance**.  

Let's customize the scatter plot by allowing you to highlight specific months with a distinct color. The selected months will appear in one color, while all other months will appear in a different color. This customization reveals whether there are **seasonal patterns** in the relationship between observed and modeled SWE, allowing us to distinguish model behavior during the key snowpack phases of <span style="color: teal;">accumulation</span> and <span style="color: tomato;">ablation (melt)</span>. Identifying these patterns is important for diagnosing the model’s strengths and limitations during different parts of the snow season.

You can change the list of highlighted months (for example, October–December for early accumulation or March–May for spring melt) to explore in the scatter plot how model performance varies across different parts of the snow season. This seasonal perspective motivates the _peak SWE analysis_ that follows.

##### 📊 For this example, let's highlight the _early snow accumulation period_ of October - January:

In [ ]:
plot = plot_utils.plot_custom_scatter_SWE(
    obs_df,
    model_df,
    f"{my_site_code}",
    f"{my_site_code}",
    site_label=my_site_code,
    highlight_months=[10, 11, 12, 1],
)

plot

<div style="color:black; background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
<h4>🧠 Reflect</h4>
<p>What does this plot tell us about how well the model performs during the early snow accumulation period at this site?<br>
HINT: How close are the <span style="color: teal;">green points</span> to the 1:1 line?</p>
</div>

## 5. Peak SWE Evaluation at the Watershed Scale  
As we saw in the previous section, how well a model matches observations can differ greatly throughout the year. The following section focuses on **peak SWE** (or maximum SWE) analysis. 

**Peak SWE is a key diagnostic for snow-dominated hydrologic systems** because it represents the maximum amount of liquid water stored in the snowpack before the spring melt. Evaluating both the magnitude (quantity) and timing (date) of peak SWE provides insight into whether the model is accurately representing snow accumulation and seasonal energy balance.  

Errors in peak SWE can have important hydrologic consequences, as peak accumulation strongly influences:
- The volume of water available for spring runoff
- The timing of streamflow peaks
- Soil moisture recharge and groundwater contributions

<div style="padding: 10px;">
  <img src="../img/peak_swe_example.png" style="width:85%;">
</div>  

_Example daily SWE at a single site, showing two important periods in snow processes: accumulation (before peak) and ablation (after peak). The vertical line marks peak SWE._

### 5.1 Comparing Modeled and Observed Peak SWE at All Sites in the Watershed
In this section, we evaluate observed and modeled peak SWE for all stations within our watershed and for all years selected in the `StartDate` and `EndDate` above. 

#### 📋 Modeled SWE on the Date of Observed Peak SWE (magnitude)  
This comparison evaluates the modeled SWE on the **specific day when observed SWE reaches its maximum.** By fixing the timing to the observed peak, this comparison isolates errors in SWE magnitude.  
It answers the question: *How much SWE does the model simulate on the day the observed snowpack reaches its maximum?*

In [ ]:
# isolate the columns associated with observations and model predictions.
# these will be inputs to our same-day comparison function.
obs_swe_cols = obs_df.columns.tolist()
mod_swe_cols = model_df.columns.tolist()

Use the `modeled_swe_at_observed_peak` utility function to perform the same-day SWE comparison.

In [ ]:
# compute the same-day SWE comparison during the observed peak SWE for each of the observation and modeled sites
df_observed_peak = snow_utils.modeled_swe_at_observed_peak(obs_df, model_df)
df_observed_peak

##### 📊 Visualize the amount of SWE on **the day of observed peak SWE occurs** for both the model and observations at each station

In [ ]:
# Rearrange the dataframe to long format for easier plotting
df_long = (
    df_observed_peak
    .reset_index()  
    .melt(
        id_vars=['Station', 'Water_Year', 'date'],
        value_vars=['Observed', 'Modeled'],
        var_name='Source',
        value_name='SWE'
    )
)
# Create scatter plot of observed and modeled SWE on the day of observed peak SWE
scatter_obs_peak = df_long.hvplot.scatter(
    x='Station',
    y='SWE',
    by='Source',  # Observed vs Modeled
    ylabel='SWE on Observed Peak Day (mm)',
    title='Observed and Modeled SWE on the Day of Observed Peak SWE',
    size=70,
    width=700,
    height=450,
    alpha=0.8,
    hover_cols=['Water_Year'],
    rot=45
)

# Customize the scatter plot appearance
scatter_by_station = (
    scatter_obs_peak  
    .opts(legend_position='top_right')
)

scatter_by_station

#### 📋 Modeled vs Observed Peak SWE Comparison (timing & magnitude)  
This comparison evaluates the modeled and observed peak SWE values and their corresponding dates independently. Unlike the previous comparison that fixed the timing to the observed peak swe, this analysis shows the actual days of modeled and observed peak SWE, which may occur on different dates. As a result, it captures errors in both **peak SWE magnitude** and **peak timing**.

In [ ]:
# compute the different-day SWE comparison for each of the observed and modeled sites.
df_both_peak = snow_utils.modeled_vs_observed_peak_swe(obs_df, model_df)
df_both_peak

##### 📊 Visualize the quantity of peak SWE for both the model and observations at each station

In [ ]:
### NEED TO DECIDE HOW TO FORMAT THIS PLOT AND IF WE WANT TO HAVE THE "SAME_DAY" PLOT

# Create the scatter plot
scatter_plot_both_peak = df_both_peak.hvplot.scatter(
    x='Observed',
    y='Modeled',
    xlabel='Observed SWE (mm)',
    ylabel='Modeled SWE (mm)',
    title='Modeled vs. Observed Peak SWE',
    size=35,
    width=500,
    height=400,
    color='#E69F00',
    hover_cols=['Station', 'Water_Year']
)#.relabel('Peak SWE')

# Add 1:1 line (perfect match line)
swe_max = df_both_peak[['Observed', 'Modeled']].max().max()

one_to_one_line = hv.Curve(([0, swe_max], [0, swe_max])).opts(
    color='gray',
    line_dash='dashed',
    line_width=1,
).relabel('1:1 Line')

# Combine scatter plot and 1:1 line into an Overlay
scatter_with_line = (scatter_plot_both_peak * one_to_one_line).opts( #scatter_plot_obs_peak * 
    legend_position='bottom_right'
)

scatter_with_line

### 5.2 Visualizing Model Error for Peak SWE

The previous scatter plots indicate that the modeled and observed peak SWE magnitude and timing don't always align. Next, we plot the degree to which   

The previous scatter plots highlight differences between modeled and observed peak SWE timing and magnitude, but interpreting these variations can be challenging when comparing modeled and observed values directly. To make these differences more explicit, we compute errors in both peak timing and peak SWE magnitude and visualize them directly. This approach clarifies both the direction and magnitude of model bias and facilitates comparison across stations and water years.

First, add columns `Peak_Date_Diff_Days` and `Peak_SWE_Diff` to the DataFrame `df_both_peak` for computed difference in peak SWE date difference and peak SWE quantity between modeled and observed:

In [ ]:
# Compute the difference in peak SWE days and peak SWE amounts between modeled and observed
df_both_peak['Peak_Date_Diff_Days'] = (df_both_peak['Modeled_Date'] - 
                                      df_both_peak['Observed_Date']).dt.days
df_both_peak['Peak_SWE_Diff'] = (df_both_peak['Modeled'] - 
                           df_both_peak['Observed'])

df_both_peak

##### 📊 Visualize the error between the modeled and observed peak SWE  

In [ ]:
# Filter to separate each water year
year1 = df_both_peak[df_both_peak['Water_Year'] == df_both_peak['Water_Year'].min()]
year2 = df_both_peak[df_both_peak['Water_Year'] == df_both_peak['Water_Year'].max()]

bar1 = year1.hvplot.bar(
    x='Station',
    y='Peak_Date_Diff_Days',
    rot=45,
    ylabel='Date Difference (days)',
    title=f'Peak SWE Date Difference {year1["Water_Year"].iloc[0]} (model - obs)',
    width=400,
    height=400,
    color='Peak_Date_Diff_Days',
    hover_cols=['Modeled', 'Observed']
)
bar2 = year1.hvplot.bar(
    x='Station',
    y='Peak_SWE_Diff',
    rot=45,
    ylabel='SWE Difference (m)',
    title=f'Peak SWE Difference {year1["Water_Year"].iloc[0]} (model - obs)',
    width=400,
    height=400,
    color='Peak_SWE_Diff',
    hover_cols=['Modeled', 'Observed']
)

# Combine side by side
layout = (bar1 + bar2)
layout.opts(shared_axes=False)

The left panel shows the timing error (date difference) and the right panel the magnitude error (SWE difference). When we computed the difference in date and SWE quantity above, we took `modeled - observed` so:  

| | DATE OF PEAK SWE | PEAK SWE QUANTITY |
|---|---|---|
| + Positive Values | modeled AFTER observed | modeled GREATER THAN observed |
| - Negative Values | modeled BEFORE observed | modeled LESS THAN observed |  

<div style="color:black; background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
<h4>🧠 Reflect</h4>
<p>Looking at the two plots, what could be some reasons for the model having simulated peak SWE both earlier and less than the observed peak SWE? Perhaps try changing the <code>my_site_code</code> from earlier in the notebook to rerun <code>plot_utils.comparison_plots()</code> to see the timeseries for a different station to look at the peak magnitude and timing. 

<br>What happens if you change the year that is plotted? <br>✏️ Try modifying the bar plot code from <code>bar1 = year1.hvplot.bar</code> to <code>bar1 = year2.hvplot.bar</code>. Don't forget to change the title!</p>
</div>

#### 📊 Next, we combine the timing and magnitude errors and plot them together for each station.

In [ ]:


scatter = df_both_peak.hvplot.scatter(
    x='Peak_Date_Diff_Days',
    y='Peak_SWE_Diff',
    by='Station', # Water_Year
    xlabel='Peak SWE Timing Error (days)',
    ylabel='Peak SWE Magnitude Error (mm)',
    title='Peak SWE Timing vs Magnitude Error',
    size=80,
    width=600,
    height=400,
    hover_cols=['Water_Year']
)

# Add reference lines
vline = hv.VLine(0).opts(color='gray', line_dash='dashed')
hline = hv.HLine(0).opts(color='gray', line_dash='dashed')

(scatter * vline * hline).opts(legend_position='bottom_left', show_grid=True)


✏️ **Try changing how we view this plot.**  
We can modify a line in the section of code from `by='Station'` to `by='Water_Year'` to better visualize the errors in the different Water Years.  
Are there any patterns that jump out? Which year was modeled peak SWE consistently less than observed peak SWE? 

### 5.3 Visualizing Model Error for Melt Period
The function `compute_melt_period_obs_vs_model` summarizes melt behavior for both the observed and modeled SWE time series at each station and for each water year, allowing the two datasets to be compared side by side. For every station-year combination, it identifies the date of peak SWE, then defines the end of the melt period as the first date after that peak when SWE reaches zero and stays at zero for at least `min_zero_days` consecutive days. Using these two dates, it computes the melt period length in days and the average melt rate over the full melt period, expressed in meters per day. The final output is a table that reports these melt timing and melt rate statistics for both observations and model simulations, making it possible to evaluate whether the model melts snow too early or too late, too quickly or too slowly, and how well it reproduces the overall timing and pace of seasonal snow disappearance across sites and years.


In [ ]:
melt_df_stats = snow_utils.compute_melt_period_obs_vs_model(obs_df, model_df, min_zero_days=10)
melt_df_stats

From a snow hydrologist’s perspective, this table is very useful because it captures three core aspects of seasonal snow behavior: **peak timing and magnitude**, **melt-out timing**, and **average melt rate**. The timeline or dumbbell view is often the most intuitive for timing diagnostics, while the 1:1 scatter is the best overall summary of performance.

In [ ]:
plot_utils.plot_scatter_melt_metrics(melt_df_stats)

While site-level comparisons provide useful detail, it is often more informative to evaluate model behavior at the **process level** to understand systematic tendencies across the watershed. In this step, we assess whether the model tends to melt snow earlier or later than observed by analyzing the difference in melt period length (model minus observed). Positive differences indicate that the model retains snow longer (later melt), while negative differences indicate earlier melt. The following visualization summarizes this behavior across all stations and water years, highlighting both the direction and magnitude of melt timing bias, and providing an overall assessment of whether the model systematically accelerates or delays snowmelt processes.

In [ ]:
# Process-level assessment
plot_utils.plot_melt_bias_summary(
    melt_df_stats,                          # Pandas DataFrame containing site melt statistics
    "Melt_Period_Days_Obs",                 # Name of the column corresponding with the observation data
    "Melt_Period_Days_Mod",                 # Name of the column corresponding with the modeled data
    "Melt Period Length (Obs vs Model)"     # Title of the plot
)

## 6. Compute and Statistics and Error Metrics  
The previous section visualized when and where modeled SWE differs from observations, both in terms of peak SWE timing and magnitude. However, visual inspection alone makes it difficult to compare performance across sites or to summarize model behavior in a consistent or quantifiable way. In this section, we compute commonly used statistical error metrics to quantify model performance, allowing us to objectively assess bias, error magnitude, and variability for sites within the watershed.  

In [ ]:
# Check for mismatched site columns between obs and model data
for col in obs_df.columns:
    if col not in model_df.columns:
        print(f"{col} missing in model data")

### 6.1 Compute Metrics

Using the modeled and observed dataframes, we can compute summary metrics. The following is some examples demonstrating how to use individual metrics of interest for a specific site, referenced as `my_site_code`.

In [ ]:
bias = metric_utils.bias(obs_df.loc[:, f'{my_site_code}'], model_df.loc[:, f'{my_site_code}'])
abs_bias = metric_utils.absolute_relative_bias(obs_df.loc[:, f'{my_site_code}'], model_df.loc[:, f'{my_site_code}'])
srho = metric_utils.spearman_rank(obs_df.loc[:, f'{my_site_code}'], model_df.loc[:, f'{my_site_code}'])

print(f"For site {my_site_code}: bias = {bias}, abs_bias = {abs_bias}, srho = {srho}")

Apply it to all sites using the `calculate_metrics` capability of the evaluation framework.

In [ ]:
metrics_df = evaluation_utils.calculate_metrics(
    obs_df,                # Pandas DataFrame containing the time series observations.    
    model_df,              # Pandas DataFrame containing the time series model calculations.
    metadata_df,                   # Pandas DataFrame containing location and site attribute information.
    metrics_list=None,             # List of metrics to calculate. None indicates that all metrics will be computed.
    write_csv=False,               # Indicates the the output should not be saved to CSV (default = False)
    csv_path=None,                 # Indicates the the output should not be saved to CSV (default = None)
)
metrics_df

Look at plots of summary statistics for each station. Here we look at Bias and NSE for each station in the watershed:

In [ ]:
# Bias scatter
scatter = metrics_df.hvplot.scatter(
    x='site_id',
    y='bias',
    size=100,
    rot=45,
    ylabel='Bias (m)',
    title='Mean SWE Bias by Station'
)

hline = hv.HLine(0).opts(color='black', line_dash='dashed', line_width=1)

scatter * hline

In [ ]:
# NSE histogram
metrics_df.hvplot.bar(
    x='site_id',
    y='nse',
    rot=45,
    ylabel='Nash–Sutcliffe Efficiency',
    title='NSE by Station',
    height=400,
    width=600,
    bar_width=0.5
)


<div style="color:black; background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
  <h4>🧠 Reflect</h4>
  <p>
    If you recall from earlier, we plotted the timeseries of our selected station. Replot it below. Do the metrics make sense given the visual comparison between modeled and observed? For example, when you look at the timeseries, is the model consistently predicting SWE to be higher or lower than observations? Does this align with the <b>Bias</b> sign (+ or -)?
  </p>
</div> 

In [ ]:
plot_utils.comparison_plots(obs_df,  model_df, f'{my_site_code}', f'{my_site_code}', site_label=None)

# Change the site code to see other Snotel Stations --> e.g., '688:CO:SNTL'
#plot_utils.comparison_plots(obs_df,  model_df, '688:CO:SNTL', '688:CO:SNTL', site_label=None)

<div style="color:black; background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
<h4>🧠 Reflect</h4>
<p>You now have several performance metrics: Bias, Pearson Correlation, Spearman Correlation, NSE, and KGE. If you had to pick just one metric to summarize model performance, which would you choose—and why?</p>
</div>

The metrics above are computed over the entire snow season, including both accumulation and ablation periods. If desired, you can subset the data and recompute the metrics to examine differences between these phases. For example, if we define the **ablation period** as April through June, the following code computes statistics for data within those months. This approach provides a more detailed understanding of how well the model represents snow accumulation versus melt processes and where performance differences may occur.

In [ ]:
ablation_months = [4, 5, 6]  # 4:April, 5:May, 6:June. 

# Create subsets of the aligned dataframes that exclude the ablation months
obs_df_ablation = obs_df[obs_df.index.month.isin(ablation_months)]
model_df_ablation = model_df[model_df.index.month.isin(ablation_months)]

# Compute metrics for the ablation period
metrics_df_ablation = evaluation_utils.calculate_metrics(
    obs_df_ablation,
    model_df_ablation,
    metadata_df,
    metrics_list=None,
    write_csv=False,
    csv_path=None,
)

metrics_df_ablation

### 6.2 Combining Magnitude (Absolute Relative Bias) and Timing (Spearman’s Rho) Using the Condon Metric 

One way to learn more about the model performance is to combine metrics that tell us about different aspects of model behavior—such as timing, variability, and magnitude—rather than relying on a single summary measure.

The Condon diagram separates model performance into quadrants based on two metrics: **Spearman’s rho** (shape/time agreement) and **relative bias** (magnitude error). The horizontal line at 0.5 distinguishes whether the model captures the temporal pattern well (above 0.5 = good shape), while the vertical line is traditionally placed at a relative bias of 1.0, which represents a 100% error. This means the model’s total error is as large as the observed signal itself. This threshold has a clear physical interpretation and is used in the original Condon framework to distinguish acceptable vs. large bias. 

In [ ]:
plot_utils.plot_condon_diagram(metrics_df, variable="SWE")